# Civitai ETL Notebook

This notebook performs an ETL process to join data from `all_comfyui.feather` with the `civitai.db` SQLite database.

In [1]:
import pandas as pd
import sqlite3
import pyarrow.feather as feather
import polars as pl
import sqlite3
import pyarrow.dataset as ds
import pandas as pd
import datetime
import pyarrow.dataset as ds
# import dask.dataframe as dd

In [2]:
# 1. 定义路径
feather_path = '/Users/lionelyip/PycharmProjects/GenImgeCrawler/data/all_comfyui.feather'
db_path = '/Users/lionelyip/PycharmProjects/GenImgeCrawler/data/comfyui_data.db'

## 1. Load Feather File

In [ ]:
# 2. 连接 SQLite 数据库
conn = sqlite3.connect(db_path)

# 3. 创建 PyArrow Dataset 对象 (瞬间完成，不占内存)
dataset = ds.dataset(feather_path, format="ipc")

# 获取总行数用于估算进度（可选）
total_rows = dataset.count_rows()
print(f"开始处理，总行数: {total_rows}")

# 4. 设置分批大小
batch_size = 1000
batch_count = 0

# 5. 开始迭代读取并写入
# to_batches 会自动处理解压，每次只吐出一块数据
for batch in dataset.to_batches(batch_size=batch_size):
    # 将这一块转为 Pandas DataFrame
    df_chunk = batch.to_pandas()

    # --- 数据清洗与格式转换 ---

    # A. 删除无用的索引列 (如果存在)
    if '__index_level_0__' in df_chunk.columns:
        df_chunk = df_chunk.drop(columns=['__index_level_0__'])

    # B. [关键步骤] 复杂类型转字符串
    # SQLite 只能存 Text/Int/Float，不能存 List/Dict
    # 根据你的数据预览，models 是 list，loras 是 dict
    complex_cols = ['models', 'loras', 'ai_json']
    for col in complex_cols:
        if col in df_chunk.columns:
            # 强制转换为字符串格式，确保能存入 SQLite
            df_chunk[col] = df_chunk[col].astype(str)

    # --- 写入数据库 ---

    # if_exists='append': 如果表存在就追加，不存在就创建
    # index=False: 不要把 Pandas 的索引存进去
    df_chunk.to_sql('comfyui_table', conn, if_exists='append', index=False)

    batch_count += 1
    rows_processed = batch_count * batch_size
    print(f"[{datetime.datetime.now().strftime('%H:%M:%S')}] 已处理批次: {batch_count} (约 {rows_processed} 行)")

    # 显式删除变量，帮助垃圾回收
    del df_chunk

# 6. 关闭连接及创建索引（可选）
print("数据插入完成，正在创建索引以加速查询...")
cursor = conn.cursor()
# 建议给常用的查询字段（如 work_id）加索引
cursor.execute("CREATE INDEX IF NOT EXISTS idx_work_id ON comfyui_table (work_id)")
conn.commit()
conn.close()

print(f"全部完成！数据库保存在: {db_path}")

开始处理，总行数: 1283003
[17:01:13] 已处理批次: 1 (约 1000 行)
[17:01:13] 已处理批次: 2 (约 2000 行)
[17:01:13] 已处理批次: 3 (约 3000 行)
[17:01:14] 已处理批次: 4 (约 4000 行)
[17:01:14] 已处理批次: 5 (约 5000 行)
[17:01:14] 已处理批次: 6 (约 6000 行)
[17:01:14] 已处理批次: 7 (约 7000 行)
[17:01:14] 已处理批次: 8 (约 8000 行)
[17:01:14] 已处理批次: 9 (约 9000 行)
[17:01:14] 已处理批次: 10 (约 10000 行)
[17:01:14] 已处理批次: 11 (约 11000 行)
[17:01:14] 已处理批次: 12 (约 12000 行)
[17:01:14] 已处理批次: 13 (约 13000 行)
[17:01:14] 已处理批次: 14 (约 14000 行)
[17:01:14] 已处理批次: 15 (约 15000 行)
[17:01:14] 已处理批次: 16 (约 16000 行)
[17:01:14] 已处理批次: 17 (约 17000 行)
[17:01:14] 已处理批次: 18 (约 18000 行)
[17:01:14] 已处理批次: 19 (约 19000 行)
[17:01:14] 已处理批次: 20 (约 20000 行)
[17:01:14] 已处理批次: 21 (约 21000 行)
[17:01:14] 已处理批次: 22 (约 22000 行)
[17:01:14] 已处理批次: 23 (约 23000 行)
[17:01:14] 已处理批次: 24 (约 24000 行)
[17:01:14] 已处理批次: 25 (约 25000 行)
[17:01:14] 已处理批次: 26 (约 26000 行)
[17:01:14] 已处理批次: 27 (约 27000 行)
[17:01:14] 已处理批次: 28 (约 28000 行)
[17:01:14] 已处理批次: 29 (约 29000 行)
[17:01:15] 已处理批次: 30 (约 30000 行)
[17:01:15]

## 2. Connect to SQLite Database

In [ ]:
db_path = '../../data/civitai.db'
conn = sqlite3.connect(db_path)

# List tables in the database
query = "SELECT name FROM sqlite_master WHERE type='table';"
df_tables = pd.read_sql_query(query, conn)
df_tables

## 3. Load Data from Database

In [ ]:
# Replace 'your_table_name' with the actual table you want to join with
table_name = 'models' # Or another table from the database
query = f"SELECT * FROM {table_name}"
df_db = pd.read_sql_query(query, conn)
df_db.head()

## 4. Join Data

In [ ]:
# Replace 'feather_key' and 'db_key' with the actual column names to join on
df_merged = pd.merge(df_feather, df_db, left_on='feather_key', right_on='db_key', how='inner')
df_merged.head()

## 5. Close Connection

In [ ]:
conn.close()